
# ECE1508 Project Notebook  
## Robust 3D Vessel Aneurysm Classification: Voxels (3D CNN) vs. Points (PointNet)

This notebook implements the project:

- **Dataset:** MedMNIST `VesselMNIST3D` (28×28×28), binary classification  
- **Pipelines:**  
  1. voxel-based **3D CNN** on the original volume  
  2. point-based **PointNet** on point clouds converted from the same volume  
- **Metrics:** **AUROC** (primary), plus accuracy, balanced accuracy, precision, recall, and F1  
- **Robustness benchmark:** clean data vs. controlled **rotation** and **additive Gaussian noise** shifts  
- **Training settings:** clean training vs. robustness-oriented training  
- **Point ablations:** `xyz` vs. `xyz+I`, and `N=512` vs. `N=1024`  
- **Stability analysis:** repeated runs with multiple random seeds and mean/std summaries  
- **Milestone 4:** optional **test-time augmentation (TTA)**

### Expected outputs
This notebook saves:
- trained checkpoints
- per-epoch histories
- robustness benchmark tables
- clean-vs-shift performance drop tables
- professional figures for AUROC / accuracy / balanced accuracy / F1
- seed-averaged summaries and plots


In [ ]:

# 1) Install dependencies
!pip -q install -U medmnist scikit-learn pandas matplotlib tqdm


In [ ]:

# 2) Imports, paths, and reproducibility
import os
import math
import json
import time
import copy
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import medmnist
from medmnist import INFO, VesselMNIST3D

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

BASE_SEED = 42

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

ROOT_DIR = Path("/content/vesselmnist3d_project")
DATA_DIR = ROOT_DIR / "data"
OUT_DIR = ROOT_DIR / "outputs"
CKPT_DIR = OUT_DIR / "checkpoints"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

for p in [ROOT_DIR, DATA_DIR, OUT_DIR, CKPT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

info = INFO["vesselmnist3d"]
print(json.dumps({
    "python_class": info["python_class"],
    "task": info["task"],
    "labels": info["label"],
    "n_samples": info["n_samples"],
    "n_channels": info["n_channels"],
}, indent=2))


In [ ]:

# 3) Project configuration
@dataclass
class TrainConfig:
    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 25
    lr: float = 1e-3
    weight_decay: float = 1e-4
    patience: int = 6
    use_amp: bool = True
    robust_train: bool = False
    train_rotation_deg: float = 25.0
    train_noise_std: float = 0.08
    robust_ramp_epochs: int = 6
    clean_mix_ratio: float = 0.35
    label_smoothing: float = 0.02
    grad_clip_norm: float = 1.0

@dataclass
class PointConfig:
    num_points: int = 512
    include_intensity: bool = True
    threshold: float = 0.10
    min_candidate_ratio: float = 0.25
    weighted_sampling: bool = True

@dataclass
class TTAConfig:
    enabled: bool = False
    reps: int = 8
    rotation_deg: float = 10.0
    noise_std: float = 0.03

BASE_TRAIN_CFG = TrainConfig()
ROBUST_TRAIN_CFG = TrainConfig(robust_train=True)

# Main PointNet comparison uses xyz+I with 512 points.
POINT_MAIN_CFG = PointConfig(num_points=512, include_intensity=True)
POINT_ABLATIONS = [
    PointConfig(num_points=512, include_intensity=False),
    PointConfig(num_points=512, include_intensity=True),
    PointConfig(num_points=1024, include_intensity=True),
]

ROTATION_SEVERITIES = [0.0, 15.0, 30.0, 45.0]
NOISE_SEVERITIES = [0.0, 0.03, 0.06, 0.10]
SEED_LIST = [42, 52, 62]

MAIN_EXPERIMENTS = [
    "voxel_clean",
    "voxel_robust",
    "point_clean_xyzI_512",
    "point_robust_xyzI_512",
]

DISPLAY_NAME_MAP = {
    "voxel_clean": "Voxel-CNN (clean)",
    "voxel_robust": "Voxel-CNN (robust)",
    "voxel_robust_tta": "Voxel-CNN (robust + TTA)",
    "point_clean_xyzI_512": "PointNet xyz+I, 512 (clean)",
    "point_robust_xyzI_512": "PointNet xyz+I, 512 (robust)",
    "point_robust_xyzI_512_tta": "PointNet xyz+I, 512 (robust + TTA)",
    "point_ablation_xyz_512": "PointNet xyz, 512",
    "point_ablation_xyzI_512": "PointNet xyz+I, 512",
    "point_ablation_xyzI_1024": "PointNet xyz+I, 1024",
}

COLOR_MAP = {
    "voxel_clean": "#1f77b4",
    "voxel_robust": "#0b4ea2",
    "voxel_robust_tta": "#08306b",
    "point_clean_xyzI_512": "#d62728",
    "point_robust_xyzI_512": "#ff7f0e",
    "point_robust_xyzI_512_tta": "#b85c00",
    "point_ablation_xyz_512": "#2ca02c",
    "point_ablation_xyzI_512": "#9467bd",
    "point_ablation_xyzI_1024": "#8c564b",
}

LINESTYLE_MAP = {
    "voxel_clean": "--",
    "voxel_robust": "-",
    "voxel_robust_tta": ":",
    "point_clean_xyzI_512": "--",
    "point_robust_xyzI_512": "-",
    "point_robust_xyzI_512_tta": ":",
    "point_ablation_xyz_512": "-.",
    "point_ablation_xyzI_512": "-",
    "point_ablation_xyzI_1024": "-.",
}

METRIC_LABELS = {
    "auroc": "AUROC",
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1 Score",
}

print("Base train config:", asdict(BASE_TRAIN_CFG))
print("Robust train config:", asdict(ROBUST_TRAIN_CFG))
print("Main point config:", asdict(POINT_MAIN_CFG))
print("Seed list:", SEED_LIST)


In [ ]:

# 4) Dataset wrapper (raw volumes only)
class VesselRawVolumeDataset(Dataset):
    def __init__(self, split: str, root: str, download: bool = True):
        self.base = VesselMNIST3D(split=split, root=root, download=download, size=28)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        volume, label = self.base[idx]
        volume = torch.tensor(volume, dtype=torch.float32)
        label = int(np.asarray(label).squeeze())
        return volume, label

    @property
    def labels(self):
        return np.asarray(self.base.labels).astype(int).reshape(-1)

train_ds = VesselRawVolumeDataset("train", str(DATA_DIR), download=True)
val_ds   = VesselRawVolumeDataset("val",   str(DATA_DIR), download=True)
test_ds  = VesselRawVolumeDataset("test",  str(DATA_DIR), download=True)

print("Split sizes:", len(train_ds), len(val_ds), len(test_ds))
print("Train label counts:", pd.Series(train_ds.labels).value_counts().sort_index().to_dict())
print("Val label counts:", pd.Series(val_ds.labels).value_counts().sort_index().to_dict())
print("Test label counts:", pd.Series(test_ds.labels).value_counts().sort_index().to_dict())

label_counts = np.bincount(train_ds.labels, minlength=2)
class_weights_np = label_counts.sum() / (len(label_counts) * np.maximum(label_counts, 1))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
print("Class weights:", class_weights)

def make_loader(ds, batch_size, shuffle=False, num_workers=0, seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)
    kwargs = dict(
        dataset=ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
        generator=generator,
    )
    if num_workers > 0:
        kwargs["persistent_workers"] = True
    return DataLoader(**kwargs)

def build_loaders(train_cfg: TrainConfig, seed: int = 42):
    train_loader = make_loader(train_ds, train_cfg.batch_size, shuffle=True,  num_workers=train_cfg.num_workers, seed=seed)
    val_loader   = make_loader(val_ds,   train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers, seed=seed)
    test_loader  = make_loader(test_ds,  train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers, seed=seed)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_loaders(BASE_TRAIN_CFG, seed=BASE_SEED)


In [ ]:

# 5) Quick data sanity check / visualization

def show_middle_slices(dataset, n=4):
    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))
    for row in range(n):
        volume, label = dataset[row]
        vol = volume[0].numpy()
        z, y, x = vol.shape[0]//2, vol.shape[1]//2, vol.shape[2]//2
        axes[row, 0].imshow(vol[z], cmap="gray")
        axes[row, 0].set_title(f"Axial | y={label}")
        axes[row, 1].imshow(vol[:, y, :], cmap="gray")
        axes[row, 1].set_title("Coronal")
        axes[row, 2].imshow(vol[:, :, x], cmap="gray")
        axes[row, 2].set_title("Sagittal")
        for c in range(3):
            axes[row, c].axis("off")
    plt.tight_layout()
    plt.show()

show_middle_slices(train_ds, n=4)



## Why the robustness benchmark matters

- **Rotation** approximates orientation variability and misalignment in 3D acquisition.
- **Additive Gaussian noise** approximates acquisition noise and reconstruction variability.
- Applying the **same perturbation protocol** to voxel and point pipelines keeps the representation comparison fair.


In [ ]:

# 6) 3D corruption / augmentation utilities

def _rotation_matrix_xyz(rx, ry, rz, device):
    cx, sx = torch.cos(rx), torch.sin(rx)
    cy, sy = torch.cos(ry), torch.sin(ry)
    cz, sz = torch.cos(rz), torch.sin(rz)

    Rx = torch.tensor([
        [1.0, 0.0, 0.0],
        [0.0, cx.item(), -sx.item()],
        [0.0, sx.item(),  cx.item()],
    ], dtype=torch.float32, device=device)

    Ry = torch.tensor([
        [ cy.item(), 0.0, sy.item()],
        [ 0.0,       1.0, 0.0],
        [-sy.item(), 0.0, cy.item()],
    ], dtype=torch.float32, device=device)

    Rz = torch.tensor([
        [cz.item(), -sz.item(), 0.0],
        [sz.item(),  cz.item(), 0.0],
        [0.0,        0.0,       1.0],
    ], dtype=torch.float32, device=device)

    return Rz @ Ry @ Rx


def apply_random_rotation_3d(volumes: torch.Tensor, max_degrees: float) -> torch.Tensor:
    if max_degrees <= 0:
        return volumes

    B = volumes.shape[0]
    device = volumes.device
    theta = torch.zeros((B, 3, 4), dtype=torch.float32, device=device)

    for i in range(B):
        angles_deg = torch.empty(3, device=device).uniform_(-max_degrees, max_degrees)
        angles_rad = angles_deg * math.pi / 180.0
        R = _rotation_matrix_xyz(angles_rad[0], angles_rad[1], angles_rad[2], device)
        theta[i, :, :3] = R

    grid = F.affine_grid(theta, size=volumes.size(), align_corners=False)
    rotated = F.grid_sample(
        volumes,
        grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=False,
    )
    return rotated.clamp(0.0, 1.0)


def apply_gaussian_noise(volumes: torch.Tensor, std: float) -> torch.Tensor:
    if std <= 0:
        return volumes
    noisy = volumes + torch.randn_like(volumes) * std
    return noisy.clamp(0.0, 1.0)


def apply_shift(volumes: torch.Tensor, rotation_deg: float = 0.0, noise_std: float = 0.0) -> torch.Tensor:
    shifted = volumes
    if rotation_deg > 0:
        shifted = apply_random_rotation_3d(shifted, rotation_deg)
    if noise_std > 0:
        shifted = apply_gaussian_noise(shifted, noise_std)
    return shifted


def apply_train_augmentation(volumes: torch.Tensor, cfg: TrainConfig, epoch_idx: int) -> torch.Tensor:
    if not cfg.robust_train:
        return volumes

    out = volumes.clone()
    ramp = min(1.0, epoch_idx / max(cfg.robust_ramp_epochs, 1))
    rot = cfg.train_rotation_deg * ramp
    noise = cfg.train_noise_std * ramp

    corrupt_mask = torch.rand(out.shape[0], device=out.device) > cfg.clean_mix_ratio
    if corrupt_mask.any():
        out[corrupt_mask] = apply_shift(out[corrupt_mask], rotation_deg=rot, noise_std=noise)
    return out


In [ ]:

# 7) Voxel -> point-cloud conversion

def volume_to_point_cloud(volume: torch.Tensor, point_cfg: PointConfig) -> torch.Tensor:
    assert volume.ndim == 4 and volume.shape[0] == 1
    v = volume[0]
    D, H, W = v.shape
    flat = v.reshape(-1)

    candidate_idx = torch.nonzero(flat >= point_cfg.threshold, as_tuple=False).squeeze(1)

    min_candidates = max(32, int(point_cfg.num_points * point_cfg.min_candidate_ratio))
    if candidate_idx.numel() < min_candidates:
        k = min(max(point_cfg.num_points, min_candidates), flat.numel())
        candidate_idx = torch.topk(flat, k=k, largest=True).indices

    cand_vals = flat[candidate_idx].clamp_min(1e-6)

    if candidate_idx.numel() >= point_cfg.num_points:
        if point_cfg.weighted_sampling:
            chosen_local = torch.multinomial(cand_vals, point_cfg.num_points, replacement=False)
        else:
            perm = torch.randperm(candidate_idx.numel(), device=volume.device)[:point_cfg.num_points]
            chosen_local = perm
        chosen_idx = candidate_idx[chosen_local]
    else:
        if point_cfg.weighted_sampling:
            chosen_local = torch.multinomial(cand_vals, point_cfg.num_points, replacement=True)
        else:
            chosen_local = torch.randint(0, candidate_idx.numel(), (point_cfg.num_points,), device=volume.device)
        chosen_idx = candidate_idx[chosen_local]

    z = chosen_idx // (H * W)
    rem = chosen_idx % (H * W)
    y = rem // W
    x = rem % W

    coords = torch.stack([x, y, z], dim=1).float()
    denom = torch.tensor([max(W - 1, 1), max(H - 1, 1), max(D - 1, 1)], device=coords.device).float()
    coords = 2.0 * (coords / denom) - 1.0

    coords = coords - coords.mean(dim=0, keepdim=True)
    scale = coords.norm(dim=1).max().clamp_min(1e-6)
    coords = coords / scale

    if point_cfg.include_intensity:
        intensity = flat[chosen_idx].unsqueeze(1)
        points = torch.cat([coords, intensity], dim=1)
    else:
        points = coords

    return points


def volumes_to_point_batch(volumes: torch.Tensor, point_cfg: PointConfig) -> torch.Tensor:
    pts = [volume_to_point_cloud(volumes[i], point_cfg) for i in range(volumes.shape[0])]
    return torch.stack(pts, dim=0)

sample_volume, sample_label = train_ds[0]
sample_points = volume_to_point_cloud(sample_volume, POINT_MAIN_CFG)
print("Sample volume shape:", tuple(sample_volume.shape))
print("Sample point cloud shape:", tuple(sample_points.shape))


In [ ]:

# 8) Model definitions
class VoxelCNN3D(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),

            nn.Conv3d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class TransformNet(nn.Module):
    def __init__(self, k: int = 3):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Conv1d(k, 64, 1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, 1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Linear(64, k * k),
        )
        nn.init.zeros_(self.fc[-1].weight)
        nn.init.zeros_(self.fc[-1].bias)

    def forward(self, x):
        B = x.shape[0]
        feat = self.mlp(x)
        feat = torch.max(feat, dim=2).values
        mat = self.fc(feat)
        identity = torch.eye(self.k, device=x.device, dtype=x.dtype).view(1, self.k * self.k).repeat(B, 1)
        mat = mat + identity
        return mat.view(B, self.k, self.k)


class PointNetClassifier(nn.Module):
    def __init__(self, input_dim: int = 4, num_classes: int = 2, use_input_transform: bool = True):
        super().__init__()
        self.use_input_transform = use_input_transform
        self.input_tnet = TransformNet(k=3) if use_input_transform else None

        self.shared_mlp = nn.Sequential(
            nn.Conv1d(input_dim, 64, 1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, 1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        if self.use_input_transform:
            xyz = x[:, :3, :]
            T = self.input_tnet(xyz)
            xyz = torch.bmm(T, xyz)
            if x.shape[1] > 3:
                x = torch.cat([xyz, x[:, 3:, :]], dim=1)
            else:
                x = xyz

        feat = self.shared_mlp(x)
        global_feat = torch.max(feat, dim=2).values
        return self.classifier(global_feat)


def build_model(pipeline: str, point_cfg: Optional[PointConfig] = None) -> nn.Module:
    if pipeline == "voxel":
        return VoxelCNN3D(num_classes=2).to(DEVICE)
    if pipeline == "point":
        assert point_cfg is not None
        input_dim = 4 if point_cfg.include_intensity else 3
        return PointNetClassifier(input_dim=input_dim, num_classes=2, use_input_transform=True).to(DEVICE)
    raise ValueError(f"Unknown pipeline: {pipeline}")


In [ ]:

# 9) Prediction, threshold calibration, training, and evaluation utilities

def prepare_inputs(volumes: torch.Tensor, pipeline: str, point_cfg: Optional[PointConfig] = None) -> torch.Tensor:
    if pipeline == "voxel":
        return volumes
    if pipeline == "point":
        assert point_cfg is not None
        return volumes_to_point_batch(volumes, point_cfg)
    raise ValueError(f"Unknown pipeline: {pipeline}")


def probs_from_logits(logits: torch.Tensor) -> torch.Tensor:
    return torch.softmax(logits, dim=1)[:, 1]


def build_criterion():
    return nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=float(BASE_TRAIN_CFG.label_smoothing),
    )


@torch.no_grad()
def collect_predictions(
    model: nn.Module,
    loader: DataLoader,
    pipeline: str,
    point_cfg: Optional[PointConfig] = None,
    corruption: Optional[dict] = None,
    tta_cfg: Optional[TTAConfig] = None,
):
    model.eval()
    criterion = build_criterion()
    all_labels, all_probs, all_losses = [], [], []

    for volumes, labels in tqdm(loader, desc=f"Eval [{pipeline}]", leave=False):
        volumes = volumes.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        reps = tta_cfg.reps if (tta_cfg is not None and tta_cfg.enabled) else 1
        probs_accum = 0.0
        loss_accum = 0.0

        for _ in range(reps):
            v = volumes.clone()

            if corruption is not None:
                v = apply_shift(
                    v,
                    rotation_deg=float(corruption.get("rotation_deg", 0.0)),
                    noise_std=float(corruption.get("noise_std", 0.0)),
                )

            if tta_cfg is not None and tta_cfg.enabled:
                v = apply_shift(v, rotation_deg=tta_cfg.rotation_deg, noise_std=tta_cfg.noise_std)

            x = prepare_inputs(v, pipeline, point_cfg)
            logits = model(x)
            probs = probs_from_logits(logits)
            loss = criterion(logits, labels)

            probs_accum = probs_accum + probs
            loss_accum = loss_accum + loss.detach()

        probs_accum = probs_accum / reps
        loss_accum = loss_accum / reps

        all_labels.append(labels.detach().cpu())
        all_probs.append(probs_accum.detach().cpu())
        all_losses.append(loss_accum.item())

    y_true = torch.cat(all_labels).numpy()
    y_prob = torch.cat(all_probs).numpy()
    mean_loss = float(np.mean(all_losses))
    return y_true, y_prob, mean_loss


def calibrate_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr = 0.5
    best_score = -np.inf
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = balanced_accuracy_score(y_true, y_pred)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float, loss_value: Optional[float] = None) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "loss": float(loss_value) if loss_value is not None else np.nan,
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "n_samples": int(len(y_true)),
    }
    return metrics


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    pipeline: str,
    point_cfg: Optional[PointConfig] = None,
    corruption: Optional[dict] = None,
    tta_cfg: Optional[TTAConfig] = None,
    threshold: float = 0.5,
) -> dict:
    y_true, y_prob, mean_loss = collect_predictions(
        model=model,
        loader=loader,
        pipeline=pipeline,
        point_cfg=point_cfg,
        corruption=corruption,
        tta_cfg=tta_cfg,
    )
    return compute_metrics(y_true, y_prob, threshold=threshold, loss_value=mean_loss)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    pipeline: str,
    train_cfg: TrainConfig,
    point_cfg: Optional[PointConfig] = None,
    epoch_idx: int = 1,
) -> float:
    model.train()
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=float(train_cfg.label_smoothing),
    )
    running_losses = []

    amp_enabled = bool(train_cfg.use_amp and DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    for volumes, labels in tqdm(loader, desc=f"Train [{pipeline}]", leave=False):
        volumes = volumes.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        volumes = apply_train_augmentation(volumes, train_cfg, epoch_idx=epoch_idx)
        x = prepare_inputs(volumes, pipeline, point_cfg)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            logits = model(x)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        if train_cfg.grad_clip_norm is not None and train_cfg.grad_clip_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        running_losses.append(loss.item())

    return float(np.mean(running_losses))


def fit_model(
    exp_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    pipeline: str,
    train_cfg: TrainConfig,
    point_cfg: Optional[PointConfig] = None,
    seed: int = 42,
) -> Tuple[nn.Module, pd.DataFrame, dict]:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    history = []
    best_state = None
    best_metric = -np.inf
    best_epoch = -1
    patience_counter = 0

    for epoch in range(1, train_cfg.epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            pipeline=pipeline,
            train_cfg=train_cfg,
            point_cfg=point_cfg,
            epoch_idx=epoch,
        )

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            pipeline=pipeline,
            point_cfg=point_cfg,
            corruption=None,
            tta_cfg=None,
            threshold=0.5,
        )

        scheduler.step(val_metrics["auroc"])

        row = {
            "experiment": exp_name,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_auroc": val_metrics["auroc"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"[{exp_name}] Epoch {epoch:02d}/{train_cfg.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_auroc={val_metrics['auroc']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f}"
        )

        if val_metrics["auroc"] > best_metric:
            best_metric = val_metrics["auroc"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= train_cfg.patience:
            print(f"Early stopping triggered for {exp_name} at epoch {epoch}.")
            break

    assert best_state is not None, "Training never produced a best state."
    model.load_state_dict(best_state)

    val_y_true, val_y_prob, _ = collect_predictions(
        model=model,
        loader=val_loader,
        pipeline=pipeline,
        point_cfg=point_cfg,
        corruption=None,
        tta_cfg=None,
    )
    calibrated_threshold = calibrate_threshold(val_y_true, val_y_prob)

    ckpt_path = CKPT_DIR / f"{exp_name}.pt"
    torch.save(
        {
            "experiment": exp_name,
            "pipeline": pipeline,
            "seed": seed,
            "train_cfg": asdict(train_cfg),
            "point_cfg": asdict(point_cfg) if point_cfg is not None else None,
            "best_epoch": best_epoch,
            "best_val_auroc": best_metric,
            "calibrated_threshold": calibrated_threshold,
            "state_dict": model.state_dict(),
        },
        ckpt_path,
    )

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(TABLE_DIR / f"{exp_name}_history.csv", index=False)

    summary = {
        "experiment": exp_name,
        "pipeline": pipeline,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_auroc": float(best_metric),
        "calibrated_threshold": float(calibrated_threshold),
        "checkpoint": str(ckpt_path),
    }
    return model, hist_df, summary


In [ ]:

# 10) Benchmarking and plotting utilities

def benchmark_model(
    exp_name: str,
    model: nn.Module,
    loader: DataLoader,
    pipeline: str,
    point_cfg: Optional[PointConfig] = None,
    tta_cfg: Optional[TTAConfig] = None,
    threshold: float = 0.5,
    seed: Optional[int] = None,
) -> pd.DataFrame:
    rows = []

    clean_metrics = evaluate_model(
        model=model,
        loader=loader,
        pipeline=pipeline,
        point_cfg=point_cfg,
        corruption=None,
        tta_cfg=tta_cfg,
        threshold=threshold,
    )
    rows.append({
        "experiment": exp_name,
        "pipeline": pipeline,
        "seed": seed,
        "shift_type": "clean",
        "severity": 0,
        "rotation_deg": 0.0,
        "noise_std": 0.0,
        **clean_metrics,
    })

    for deg in ROTATION_SEVERITIES[1:]:
        metrics = evaluate_model(
            model=model,
            loader=loader,
            pipeline=pipeline,
            point_cfg=point_cfg,
            corruption={"rotation_deg": deg, "noise_std": 0.0},
            tta_cfg=tta_cfg,
            threshold=threshold,
        )
        rows.append({
            "experiment": exp_name,
            "pipeline": pipeline,
            "seed": seed,
            "shift_type": "rotation",
            "severity": deg,
            "rotation_deg": deg,
            "noise_std": 0.0,
            **metrics,
        })

    for std in NOISE_SEVERITIES[1:]:
        metrics = evaluate_model(
            model=model,
            loader=loader,
            pipeline=pipeline,
            point_cfg=point_cfg,
            corruption={"rotation_deg": 0.0, "noise_std": std},
            tta_cfg=tta_cfg,
            threshold=threshold,
        )
        rows.append({
            "experiment": exp_name,
            "pipeline": pipeline,
            "seed": seed,
            "shift_type": "noise",
            "severity": std,
            "rotation_deg": 0.0,
            "noise_std": std,
            **metrics,
        })

    df = pd.DataFrame(rows)
    clean_row = df[df["shift_type"] == "clean"].iloc[0]
    for metric in ["auroc", "accuracy", "balanced_accuracy", "precision", "recall", "f1"]:
        df[f"{metric}_drop_from_clean"] = float(clean_row[metric]) - df[metric]
    return df


def _format_rotation_axis(ax):
    ax.set_xticks(ROTATION_SEVERITIES)
    ax.set_xticklabels(["0°", "15°", "30°", "45°"])
    ax.set_xlabel("Maximum random rotation (degrees)")


def _format_noise_axis(ax):
    ax.set_xticks(NOISE_SEVERITIES)
    ax.set_xticklabels([f"{x:.2f}" for x in NOISE_SEVERITIES])
    ax.set_xlabel("Gaussian noise standard deviation ($\\sigma$)")


def _apply_professional_axis_style(ax, ylabel: str, title: str):
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.grid(True, which="major", alpha=0.35, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_metric_curves(
    df: pd.DataFrame,
    metric: str,
    title_prefix: str,
    save_name: str,
    experiments: Optional[List[str]] = None,
    ylim: Optional[Tuple[float, float]] = None,
):
    plot_df = df.copy()
    if experiments is not None:
        plot_df = plot_df[plot_df["experiment"].isin(experiments)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for exp_name in plot_df["experiment"].unique():
        sub = plot_df[plot_df["experiment"] == exp_name]
        rot = sub[sub["shift_type"].isin(["clean", "rotation"])].copy().sort_values("rotation_deg")
        noi = sub[sub["shift_type"].isin(["clean", "noise"])].copy().sort_values("noise_std")

        axes[0].plot(
            rot["rotation_deg"],
            rot[metric],
            marker="o",
            markersize=6,
            linewidth=2.2,
            label=DISPLAY_NAME_MAP.get(exp_name, exp_name),
            color=COLOR_MAP.get(exp_name),
            linestyle=LINESTYLE_MAP.get(exp_name, "-"),
        )
        axes[1].plot(
            noi["noise_std"],
            noi[metric],
            marker="o",
            markersize=6,
            linewidth=2.2,
            label=DISPLAY_NAME_MAP.get(exp_name, exp_name),
            color=COLOR_MAP.get(exp_name),
            linestyle=LINESTYLE_MAP.get(exp_name, "-"),
        )

    _apply_professional_axis_style(axes[0], METRIC_LABELS[metric], f"{title_prefix}: Rotation")
    _apply_professional_axis_style(axes[1], METRIC_LABELS[metric], f"{title_prefix}: Additive Gaussian Noise")
    _format_rotation_axis(axes[0])
    _format_noise_axis(axes[1])
    if ylim is not None:
        axes[0].set_ylim(*ylim)
        axes[1].set_ylim(*ylim)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
    fig.suptitle(f"{METRIC_LABELS[metric]} under controlled distribution shift", fontsize=16, fontweight="bold", y=1.04)
    fig.tight_layout()
    path = FIG_DIR / save_name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {path}")


def plot_clean_metric_bars(df: pd.DataFrame, metrics: List[str], experiments: List[str], save_name: str):
    clean_df = df[(df["shift_type"] == "clean") & (df["experiment"].isin(experiments))].copy()
    clean_df["display_name"] = clean_df["experiment"].map(DISPLAY_NAME_MAP)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        plot_sub = clean_df.sort_values(metric, ascending=False)
        ax.bar(
            plot_sub["display_name"],
            plot_sub[metric],
            color=[COLOR_MAP.get(x, "#4c72b0") for x in plot_sub["experiment"]],
            alpha=0.9,
        )
        _apply_professional_axis_style(ax, METRIC_LABELS[metric], f"Clean-set {METRIC_LABELS[metric]}")
        ax.tick_params(axis="x", rotation=15)
        for idx, value in enumerate(plot_sub[metric]):
            ax.text(idx, value + 0.01, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

    fig.tight_layout()
    path = FIG_DIR / save_name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {path}")


def plot_average_drop_bars(df: pd.DataFrame, metric: str, experiments: List[str], save_name: str):
    sub = df[(df["shift_type"] != "clean") & (df["experiment"].isin(experiments))].copy()
    drop_col = f"{metric}_drop_from_clean"
    summary = (
        sub.groupby(["experiment", "shift_type"], as_index=False)[drop_col]
        .mean()
    )

    pivot = summary.pivot(index="experiment", columns="shift_type", values=drop_col).reindex(experiments)
    x = np.arange(len(pivot.index))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, pivot["rotation"], width=width, label="Rotation average drop", color="#4c72b0")
    ax.bar(x + width/2, pivot["noise"], width=width, label="Noise average drop", color="#dd8452")

    ax.set_xticks(x)
    ax.set_xticklabels([DISPLAY_NAME_MAP.get(exp, exp) for exp in pivot.index], rotation=10)
    _apply_professional_axis_style(ax, f"Average {METRIC_LABELS[metric]} drop from clean", f"Average robustness drop: {METRIC_LABELS[metric]}")
    ax.legend(frameon=False)

    fig.tight_layout()
    path = FIG_DIR / save_name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {path}")


def plot_tta_gain_bars(df: pd.DataFrame, metric: str, save_name: str):
    focus = df[df["shift_type"].isin(["clean", "rotation", "noise"])].copy()
    rotation_45 = focus[(focus["shift_type"] == "rotation") & (focus["rotation_deg"] == 45.0)]
    noise_010 = focus[(focus["shift_type"] == "noise") & (focus["noise_std"] == 0.10)]
    clean = focus[focus["shift_type"] == "clean"]

    rows = []
    pairs = [
        ("voxel_robust", "voxel_robust_tta", "Voxel-CNN"),
        ("point_robust_xyzI_512", "point_robust_xyzI_512_tta", "PointNet xyz+I, 512"),
    ]
    for base_exp, tta_exp, label in pairs:
        rows.append({
            "model": label,
            "scenario": "Clean",
            "gain": float(clean.loc[clean["experiment"] == tta_exp, metric].iloc[0] - clean.loc[clean["experiment"] == base_exp, metric].iloc[0]),
        })
        rows.append({
            "model": label,
            "scenario": "Rotation 45°",
            "gain": float(rotation_45.loc[rotation_45["experiment"] == tta_exp, metric].iloc[0] - rotation_45.loc[rotation_45["experiment"] == base_exp, metric].iloc[0]),
        })
        rows.append({
            "model": label,
            "scenario": "Noise σ=0.10",
            "gain": float(noise_010.loc[noise_010["experiment"] == tta_exp, metric].iloc[0] - noise_010.loc[noise_010["experiment"] == base_exp, metric].iloc[0]),
        })
    gain_df = pd.DataFrame(rows)

    scenarios = ["Clean", "Rotation 45°", "Noise σ=0.10"]
    models = gain_df["model"].unique().tolist()
    x = np.arange(len(scenarios))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, model_name in enumerate(models):
        y = gain_df[gain_df["model"] == model_name].set_index("scenario").loc[scenarios, "gain"].values
        ax.bar(x + (i - 0.5) * width, y, width=width, label=model_name)

    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    _apply_professional_axis_style(ax, f"TTA gain in {METRIC_LABELS[metric]}", f"Inference-time gain from TTA: {METRIC_LABELS[metric]}")
    ax.axhline(0.0, color="black", linewidth=1.0)
    ax.legend(frameon=False)

    fig.tight_layout()
    path = FIG_DIR / save_name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {path}")


def summarize_seed_results(seed_benchmark_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["auroc", "accuracy", "balanced_accuracy", "precision", "recall", "f1"]
    grouped = seed_benchmark_df.groupby(["experiment", "shift_type", "rotation_deg", "noise_std"], as_index=False)[metric_cols].agg(["mean", "std"])
    grouped.columns = ["_".join(col).strip("_") for col in grouped.columns.values]
    return grouped


def plot_seed_mean_std(seed_summary_df: pd.DataFrame, metric: str, experiments: List[str], save_name: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_df = seed_summary_df.copy()

    for exp_name in experiments:
        rot = plot_df[(plot_df["experiment"] == exp_name) & (plot_df["shift_type"] == "rotation")].sort_values("rotation_deg")
        clean = plot_df[(plot_df["experiment"] == exp_name) & (plot_df["shift_type"] == "clean")]
        rot_x = np.array([0.0] + rot["rotation_deg"].tolist(), dtype=float)
        rot_mean = np.array([clean[f"{metric}_mean"].iloc[0]] + rot[f"{metric}_mean"].tolist(), dtype=float)
        rot_std = np.array([clean[f"{metric}_std"].fillna(0).iloc[0]] + rot[f"{metric}_std"].fillna(0).tolist(), dtype=float)
        axes[0].plot(rot_x, rot_mean, marker="o", linewidth=2.2, color=COLOR_MAP.get(exp_name), linestyle=LINESTYLE_MAP.get(exp_name, "-"), label=DISPLAY_NAME_MAP.get(exp_name, exp_name))
        axes[0].fill_between(rot_x, rot_mean - rot_std, rot_mean + rot_std, color=COLOR_MAP.get(exp_name), alpha=0.15)

        noi = plot_df[(plot_df["experiment"] == exp_name) & (plot_df["shift_type"] == "noise")].sort_values("noise_std")
        noi_x = np.array([0.0] + noi["noise_std"].tolist(), dtype=float)
        noi_mean = np.array([clean[f"{metric}_mean"].iloc[0]] + noi[f"{metric}_mean"].tolist(), dtype=float)
        noi_std = np.array([clean[f"{metric}_std"].fillna(0).iloc[0]] + noi[f"{metric}_std"].fillna(0).tolist(), dtype=float)
        axes[1].plot(noi_x, noi_mean, marker="o", linewidth=2.2, color=COLOR_MAP.get(exp_name), linestyle=LINESTYLE_MAP.get(exp_name, "-"), label=DISPLAY_NAME_MAP.get(exp_name, exp_name))
        axes[1].fill_between(noi_x, noi_mean - noi_std, noi_mean + noi_std, color=COLOR_MAP.get(exp_name), alpha=0.15)

    _apply_professional_axis_style(axes[0], METRIC_LABELS[metric], f"Seed-averaged {METRIC_LABELS[metric]}: Rotation")
    _apply_professional_axis_style(axes[1], METRIC_LABELS[metric], f"Seed-averaged {METRIC_LABELS[metric]}: Additive Gaussian Noise")
    _format_rotation_axis(axes[0])
    _format_noise_axis(axes[1])

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
    fig.tight_layout()
    path = FIG_DIR / save_name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {path}")


In [ ]:

# 11) Main experiment runner

def run_single_experiment(
    exp_name: str,
    pipeline: str,
    train_cfg: TrainConfig,
    point_cfg: Optional[PointConfig] = None,
    use_tta: bool = False,
    train_loader: Optional[DataLoader] = None,
    val_loader: Optional[DataLoader] = None,
    test_loader: Optional[DataLoader] = None,
    seed: int = 42,
) -> Tuple[nn.Module, pd.DataFrame, pd.DataFrame, dict]:
    if train_loader is None or val_loader is None or test_loader is None:
        train_loader, val_loader, test_loader = build_loaders(train_cfg, seed=seed)

    model = build_model(pipeline, point_cfg=point_cfg)

    model, hist_df, summary = fit_model(
        exp_name=exp_name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        pipeline=pipeline,
        train_cfg=train_cfg,
        point_cfg=point_cfg,
        seed=seed,
    )

    print("Training summary:", summary)

    tta_cfg = TTAConfig(enabled=use_tta) if use_tta else TTAConfig(enabled=False)

    benchmark_df = benchmark_model(
        exp_name=exp_name,
        model=model,
        loader=test_loader,
        pipeline=pipeline,
        point_cfg=point_cfg,
        tta_cfg=tta_cfg,
        threshold=summary["calibrated_threshold"],
        seed=seed,
    )
    benchmark_path = TABLE_DIR / f"{exp_name}_benchmark.csv"
    benchmark_df.to_csv(benchmark_path, index=False)
    print(f"Saved benchmark to: {benchmark_path}")

    return model, hist_df, benchmark_df, summary



## 12) Core experiments

The next cells run:
- `voxel_clean`
- `point_clean_xyzI_512`
- `voxel_robust`
- `point_robust_xyzI_512`

Then the notebook runs point-construction ablations:
- `xyz` vs. `xyz+I`
- `N=512` vs. `N=1024`

The main voxel-vs-point comparison uses **xyz+I with 512 points** because it is the stronger point formulation in the current project.


In [ ]:

# 12A) Optional quick-run override
# Uncomment these lines for a faster smoke test before a full run.
# BASE_TRAIN_CFG.epochs = 6
# ROBUST_TRAIN_CFG.epochs = 6
# BASE_TRAIN_CFG.patience = 3
# ROBUST_TRAIN_CFG.patience = 3

print("Base config:", asdict(BASE_TRAIN_CFG))
print("Robust config:", asdict(ROBUST_TRAIN_CFG))


In [ ]:

# 12B) Train the required baseline models
all_histories = []
all_benchmarks = []
all_summaries = []
trained_models = {}

train_loader, val_loader, test_loader = build_loaders(BASE_TRAIN_CFG, seed=BASE_SEED)

voxel_clean_model, voxel_clean_hist, voxel_clean_bench, voxel_clean_summary = run_single_experiment(
    exp_name="voxel_clean",
    pipeline="voxel",
    train_cfg=BASE_TRAIN_CFG,
    point_cfg=None,
    use_tta=False,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    seed=BASE_SEED,
)
trained_models["voxel_clean"] = voxel_clean_model
all_histories.append(voxel_clean_hist)
all_benchmarks.append(voxel_clean_bench)
all_summaries.append(voxel_clean_summary)

point_clean_model, point_clean_hist, point_clean_bench, point_clean_summary = run_single_experiment(
    exp_name="point_clean_xyzI_512",
    pipeline="point",
    train_cfg=BASE_TRAIN_CFG,
    point_cfg=POINT_MAIN_CFG,
    use_tta=False,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    seed=BASE_SEED,
)
trained_models["point_clean_xyzI_512"] = point_clean_model
all_histories.append(point_clean_hist)
all_benchmarks.append(point_clean_bench)
all_summaries.append(point_clean_summary)


In [ ]:

# 12C) Train the required robust-training models
train_loader_r, val_loader_r, test_loader_r = build_loaders(ROBUST_TRAIN_CFG, seed=BASE_SEED)

voxel_robust_model, voxel_robust_hist, voxel_robust_bench, voxel_robust_summary = run_single_experiment(
    exp_name="voxel_robust",
    pipeline="voxel",
    train_cfg=ROBUST_TRAIN_CFG,
    point_cfg=None,
    use_tta=False,
    train_loader=train_loader_r,
    val_loader=val_loader_r,
    test_loader=test_loader_r,
    seed=BASE_SEED,
)
trained_models["voxel_robust"] = voxel_robust_model
all_histories.append(voxel_robust_hist)
all_benchmarks.append(voxel_robust_bench)
all_summaries.append(voxel_robust_summary)

point_robust_model, point_robust_hist, point_robust_bench, point_robust_summary = run_single_experiment(
    exp_name="point_robust_xyzI_512",
    pipeline="point",
    train_cfg=ROBUST_TRAIN_CFG,
    point_cfg=POINT_MAIN_CFG,
    use_tta=False,
    train_loader=train_loader_r,
    val_loader=val_loader_r,
    test_loader=test_loader_r,
    seed=BASE_SEED,
)
trained_models["point_robust_xyzI_512"] = point_robust_model
all_histories.append(point_robust_hist)
all_benchmarks.append(point_robust_bench)
all_summaries.append(point_robust_summary)


In [ ]:

# 13) Point pipeline ablations
point_ablation_configs = [
    ("point_ablation_xyz_512", PointConfig(num_points=512, include_intensity=False)),
    ("point_ablation_xyzI_512", PointConfig(num_points=512, include_intensity=True)),
    ("point_ablation_xyzI_1024", PointConfig(num_points=1024, include_intensity=True)),
]

for ablation_name, point_cfg in point_ablation_configs:
    print("\nRunning ablation:", ablation_name, asdict(point_cfg))
    ablation_model, ablation_hist, ablation_bench, ablation_summary = run_single_experiment(
        exp_name=ablation_name,
        pipeline="point",
        train_cfg=BASE_TRAIN_CFG,
        point_cfg=point_cfg,
        use_tta=False,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        seed=BASE_SEED,
    )
    trained_models[ablation_name] = ablation_model
    all_histories.append(ablation_hist)
    all_benchmarks.append(ablation_bench)
    all_summaries.append(ablation_summary)


In [ ]:

# 14) Aggregate training histories / benchmarks / summaries
history_df = pd.concat(all_histories, ignore_index=True)
benchmark_df = pd.concat(all_benchmarks, ignore_index=True)
summary_df = pd.DataFrame(all_summaries)

history_df.to_csv(TABLE_DIR / "all_histories.csv", index=False)
benchmark_df.to_csv(TABLE_DIR / "all_benchmarks.csv", index=False)
summary_df.to_csv(TABLE_DIR / "training_summaries.csv", index=False)

print("History table saved to:", TABLE_DIR / "all_histories.csv")
print("Benchmark table saved to:", TABLE_DIR / "all_benchmarks.csv")
print("Training summary table saved to:", TABLE_DIR / "training_summaries.csv")

display(summary_df)
display(benchmark_df.head())


In [ ]:

# 15) Professional robustness plots for the main voxel-vs-point comparison
plot_metric_curves(
    benchmark_df,
    metric="auroc",
    title_prefix="Main comparison AUROC",
    save_name="main_comparison_auroc_curves.png",
    experiments=MAIN_EXPERIMENTS,
    ylim=(0.45, 1.0),
)

plot_metric_curves(
    benchmark_df,
    metric="accuracy",
    title_prefix="Main comparison Accuracy",
    save_name="main_comparison_accuracy_curves.png",
    experiments=MAIN_EXPERIMENTS,
    ylim=(0.0, 1.0),
)

plot_metric_curves(
    benchmark_df,
    metric="balanced_accuracy",
    title_prefix="Main comparison Balanced Accuracy",
    save_name="main_comparison_balanced_accuracy_curves.png",
    experiments=MAIN_EXPERIMENTS,
    ylim=(0.0, 1.0),
)

plot_metric_curves(
    benchmark_df,
    metric="f1",
    title_prefix="Main comparison F1",
    save_name="main_comparison_f1_curves.png",
    experiments=MAIN_EXPERIMENTS,
    ylim=(0.0, 1.0),
)


In [ ]:

# 16) Additional figures for reporting
plot_clean_metric_bars(
    benchmark_df,
    metrics=["auroc", "accuracy", "balanced_accuracy", "f1"],
    experiments=MAIN_EXPERIMENTS,
    save_name="clean_main_metrics_bars.png",
)

plot_average_drop_bars(
    benchmark_df,
    metric="auroc",
    experiments=MAIN_EXPERIMENTS,
    save_name="average_auroc_drop_main_models.png",
)

plot_average_drop_bars(
    benchmark_df,
    metric="accuracy",
    experiments=MAIN_EXPERIMENTS,
    save_name="average_accuracy_drop_main_models.png",
)


In [ ]:

# 17) Summary tables: clean performance and robustness drop
clean_summary = benchmark_df[benchmark_df["shift_type"] == "clean"].copy()
clean_summary = clean_summary.sort_values(["auroc", "accuracy"], ascending=False)

metric_drop_cols = [
    "auroc_drop_from_clean",
    "accuracy_drop_from_clean",
    "balanced_accuracy_drop_from_clean",
    "precision_drop_from_clean",
    "recall_drop_from_clean",
    "f1_drop_from_clean",
]

drop_summary = (
    benchmark_df[benchmark_df["shift_type"] != "clean"]
    .groupby(["experiment", "pipeline", "shift_type"], as_index=False)[metric_drop_cols]
    .mean()
    .sort_values(["shift_type", "auroc_drop_from_clean"])
)

clean_summary.to_csv(TABLE_DIR / "clean_summary.csv", index=False)
drop_summary.to_csv(TABLE_DIR / "robustness_drop_summary.csv", index=False)

print("=== Clean summary ===")
display(clean_summary[[
    "experiment", "pipeline", "threshold", "auroc", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "loss"
]])

print("=== Average robustness drop from clean ===")
display(drop_summary)



## 18) Optional Milestone 4: Test-time augmentation (TTA)

The next cells evaluate TTA on the main robust models:
- `voxel_robust + TTA`
- `point_robust_xyzI_512 + TTA`

TTA averages predictions over several additional rotation+noise views at test time.


In [ ]:

# 18A) Run TTA on the main robust models
tta_benchmarks = []

tta_cfg = TTAConfig(enabled=True, reps=8, rotation_deg=10.0, noise_std=0.03)
print("Using TTA config:", asdict(tta_cfg))

voxel_threshold = float(summary_df.loc[summary_df["experiment"] == "voxel_robust", "calibrated_threshold"].iloc[0])
point_threshold = float(summary_df.loc[summary_df["experiment"] == "point_robust_xyzI_512", "calibrated_threshold"].iloc[0])

voxel_robust_tta_df = benchmark_model(
    exp_name="voxel_robust_tta",
    model=trained_models["voxel_robust"],
    loader=test_loader_r,
    pipeline="voxel",
    point_cfg=None,
    tta_cfg=tta_cfg,
    threshold=voxel_threshold,
    seed=BASE_SEED,
)
tta_benchmarks.append(voxel_robust_tta_df)

point_robust_tta_df = benchmark_model(
    exp_name="point_robust_xyzI_512_tta",
    model=trained_models["point_robust_xyzI_512"],
    loader=test_loader_r,
    pipeline="point",
    point_cfg=POINT_MAIN_CFG,
    tta_cfg=tta_cfg,
    threshold=point_threshold,
    seed=BASE_SEED,
)
tta_benchmarks.append(point_robust_tta_df)

tta_df = pd.concat(tta_benchmarks, ignore_index=True)
tta_df.to_csv(TABLE_DIR / "tta_benchmarks.csv", index=False)

display(tta_df)


In [ ]:

# 18B) Compare no-TTA vs TTA for the robust models
comparison_df = pd.concat([
    benchmark_df[benchmark_df["experiment"].isin(["voxel_robust", "point_robust_xyzI_512"])],
    tta_df,
], ignore_index=True)
comparison_df.to_csv(TABLE_DIR / "robust_vs_tta_comparison.csv", index=False)

plot_metric_curves(
    comparison_df,
    metric="auroc",
    title_prefix="Robust models: AUROC with and without TTA",
    save_name="robust_vs_tta_auroc.png",
    experiments=["voxel_robust", "voxel_robust_tta", "point_robust_xyzI_512", "point_robust_xyzI_512_tta"],
    ylim=(0.45, 1.0),
)

plot_metric_curves(
    comparison_df,
    metric="accuracy",
    title_prefix="Robust models: Accuracy with and without TTA",
    save_name="robust_vs_tta_accuracy.png",
    experiments=["voxel_robust", "voxel_robust_tta", "point_robust_xyzI_512", "point_robust_xyzI_512_tta"],
    ylim=(0.0, 1.0),
)

plot_tta_gain_bars(
    comparison_df,
    metric="auroc",
    save_name="tta_gain_auroc.png",
)

plot_tta_gain_bars(
    comparison_df,
    metric="accuracy",
    save_name="tta_gain_accuracy.png",
)



## 19) Multi-seed stability analysis

The next cells rerun the most important comparisons with multiple random seeds and summarize the mean and standard deviation.


In [ ]:

# 19A) Multi-seed benchmark for the four main models
SEED_STUDY_EPOCHS = None
seed_histories = []
seed_benchmarks = []
seed_summaries = []

for seed in SEED_LIST:
    print(f"\n===== Running seed study for seed {seed} =====")
    seed_everything(seed)

    base_cfg = copy.deepcopy(BASE_TRAIN_CFG)
    robust_cfg = copy.deepcopy(ROBUST_TRAIN_CFG)
    if SEED_STUDY_EPOCHS is not None:
        base_cfg.epochs = SEED_STUDY_EPOCHS
        robust_cfg.epochs = SEED_STUDY_EPOCHS

    base_train_loader, base_val_loader, base_test_loader = build_loaders(base_cfg, seed=seed)
    robust_train_loader, robust_val_loader, robust_test_loader = build_loaders(robust_cfg, seed=seed)

    seed_specs = [
        ("voxel_clean", "voxel", base_cfg, None, base_train_loader, base_val_loader, base_test_loader),
        ("voxel_robust", "voxel", robust_cfg, None, robust_train_loader, robust_val_loader, robust_test_loader),
        ("point_clean_xyzI_512", "point", base_cfg, POINT_MAIN_CFG, base_train_loader, base_val_loader, base_test_loader),
        ("point_robust_xyzI_512", "point", robust_cfg, POINT_MAIN_CFG, robust_train_loader, robust_val_loader, robust_test_loader),
    ]

    for exp_name, pipeline, cfg, point_cfg, tr_loader, va_loader, te_loader in seed_specs:
        seeded_exp_name = f"{exp_name}_seed{seed}"
        model, hist_df, bench_df, summary = run_single_experiment(
            exp_name=seeded_exp_name,
            pipeline=pipeline,
            train_cfg=cfg,
            point_cfg=point_cfg,
            use_tta=False,
            train_loader=tr_loader,
            val_loader=va_loader,
            test_loader=te_loader,
            seed=seed,
        )
        hist_df["seed_model_name"] = hist_df["experiment"]
        hist_df["experiment"] = exp_name
        bench_df["seed_model_name"] = bench_df["experiment"]
        bench_df["experiment"] = exp_name
        summary["seed_model_name"] = summary["experiment"]
        summary["experiment"] = exp_name
        seed_histories.append(hist_df)
        seed_benchmarks.append(bench_df)
        seed_summaries.append(summary)

seed_history_df = pd.concat(seed_histories, ignore_index=True)
seed_benchmark_df = pd.concat(seed_benchmarks, ignore_index=True)
seed_summary_df = pd.DataFrame(seed_summaries)

seed_history_df.to_csv(TABLE_DIR / "seed_histories.csv", index=False)
seed_benchmark_df.to_csv(TABLE_DIR / "seed_benchmark_raw.csv", index=False)
seed_summary_df.to_csv(TABLE_DIR / "seed_training_summaries.csv", index=False)

print("Saved seed study tables.")
display(seed_summary_df.head())


In [ ]:

# 19B) Aggregate seed results and plot mean ± std
seed_summary_agg = summarize_seed_results(seed_benchmark_df)
seed_summary_agg.to_csv(TABLE_DIR / "seed_benchmark_mean_std.csv", index=False)

display(seed_summary_agg.head())

plot_seed_mean_std(
    seed_summary_agg,
    metric="auroc",
    experiments=MAIN_EXPERIMENTS,
    save_name="seed_mean_std_auroc.png",
)

plot_seed_mean_std(
    seed_summary_agg,
    metric="accuracy",
    experiments=MAIN_EXPERIMENTS,
    save_name="seed_mean_std_accuracy.png",
)


In [ ]:

# 20) Compact experiment summary JSON
artifact = {
    "dataset": "VesselMNIST3D",
    "train_config_base": asdict(BASE_TRAIN_CFG),
    "train_config_robust": asdict(ROBUST_TRAIN_CFG),
    "point_main_config": asdict(POINT_MAIN_CFG),
    "rotation_severities_degrees": ROTATION_SEVERITIES,
    "noise_severities_std": NOISE_SEVERITIES,
    "seed_list": SEED_LIST,
    "output_dir": str(OUT_DIR),
    "tables": sorted([str(p) for p in TABLE_DIR.glob("*.csv")]),
    "figures": sorted([str(p) for p in FIG_DIR.glob("*.png")]),
    "checkpoints": sorted([str(p) for p in CKPT_DIR.glob("*.pt")]),
}
with open(OUT_DIR / "run_artifact_summary.json", "w") as f:
    json.dump(artifact, f, indent=2)

print(json.dumps(artifact, indent=2))



## 21) References

[1] **PointNet**  
Qi, Su, Mo, and Guibas. *PointNet: Deep Learning on Point Sets for 3D Classification and Segmentation.* CVPR 2017.

[2] **MedMNIST / VesselMNIST3D**  
Yang et al. *MedMNIST v2: A Large-Scale Lightweight Benchmark for 2D and 3D Biomedical Image Classification.* Scientific Data 2023.


In [ ]:

# 22) Download all results, tables, checkpoints, and figures
import shutil
from google.colab import files

archive_base = '/content/vesselmnist3d_project_results'
archive_file = shutil.make_archive(archive_base, 'zip', root_dir=str(OUT_DIR))
print(f"Created archive: {archive_file}")
files.download(archive_file)
